# LaneSegNet: Complete Training Notebook

This notebook contains the full LaneSegNet model architecture and training pipeline integrated end-to-end for autonomous driving lane segmentation.

In [ ]:
# Setup and Imports
import os
import sys
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import warnings
from pathlib import Path

# MMDetection and related imports
import mmcv
from mmcv import Config, DictAction
from mmcv.cnn import Linear, build_activation_layer, ConvModule
from mmcv.runner import auto_fp16, force_fp32, get_dist_info, init_dist, get_root_logger
from mmcv.cnn.bricks.registry import TRANSFORMER_LAYER_SEQUENCE, TRANSFORMER_LAYER
from mmcv.cnn.bricks.transformer import TransformerLayerSequence, BaseTransformerLayer

from mmdet import __version__ as mmdet_version
from mmdet3d import __version__ as mmdet3d_version
from mmdet.models import DETECTORS, HEADS
from mmdet.models.builder import build_head, build_backbone, build_neck, build_loss
from mmdet.models.utils import build_transformer
from mmdet.models.utils.transformer import inverse_sigmoid
from mmdet.core import build_assigner, build_sampler, multi_apply, reduce_mean
from mmdet.apis import set_random_seed
from mmdet3d.models.detectors.mvx_two_stage import MVXTwoStageDetector
from mmdet3d.apis import init_random_seed, train_model
from mmdet3d.datasets import build_dataset
from mmdet3d.models import build_model
from mmdet3d.utils import collect_env
from mmseg import __version__ as mmseg_version

try:
    from mmdet.utils import setup_multi_processes
except ImportError:
    from mmdet3d.utils import setup_multi_processes

print("All imports successful!")

In [ ]:
# Model Architecture - Decoder Components

@TRANSFORMER_LAYER_SEQUENCE.register_module()
class LaneSegNetDecoder(TransformerLayerSequence):
    """LaneSegNet decoder with multi-layer transformer and reference point updates."""

    def __init__(self, 
                 *args, 
                 return_intermediate=False, 
                 pc_range=None, 
                 sample_idx=[0, 3, 6, 9],
                 pts_dim=3, 
                 **kwargs):
        super(LaneSegNetDecoder, self).__init__(*args, **kwargs)
        self.return_intermediate = return_intermediate
        self.fp16_enabled = False
        self.pc_range = pc_range
        self.sample_idx = sample_idx
        self.pts_dim = pts_dim

    def forward(self, query, *args, reference_points=None, reg_branches=None, key_padding_mask=None, **kwargs):
        output = query
        intermediate = []
        intermediate_reference_points = []
        lane_ref_points = reference_points[:, :, self.sample_idx * 2, :]
        
        for lid, layer in enumerate(self.layers):
            reference_points_input = lane_ref_points[..., :2].unsqueeze(2)
            output = layer(output, *args, reference_points=reference_points_input, key_padding_mask=key_padding_mask, **kwargs)
            output = output.permute(1, 0, 2)

            if reg_branches is not None:
                reg_center = reg_branches[0]
                reg_offset = reg_branches[1]

                tmp = reg_center[lid](output)
                bs, num_query, _ = tmp.shape
                tmp = tmp.view(bs, num_query, -1, self.pts_dim)
                assert reference_points.shape[-1] == self.pts_dim
                tmp = tmp + inverse_sigmoid(reference_points)
                tmp = tmp.sigmoid()
                reference_points = tmp.detach()

                coord = tmp.clone()
                coord[..., 0] = coord[..., 0] * (self.pc_range[3] - self.pc_range[0]) + self.pc_range[0]
                coord[..., 1] = coord[..., 1] * (self.pc_range[4] - self.pc_range[1]) + self.pc_range[1]
                if self.pts_dim == 3:
                    coord[..., 2] = coord[..., 2] * (self.pc_range[5] - self.pc_range[2]) + self.pc_range[2]
                centerline = coord.view(bs, num_query, -1).contiguous()

                offset = reg_offset[lid](output)
                left_laneline = centerline + offset
                right_laneline = centerline - offset
                left_laneline = left_laneline.view(bs, num_query, -1, self.pts_dim)[:, :, self.sample_idx, :]
                right_laneline = right_laneline.view(bs, num_query, -1, self.pts_dim)[:, :, self.sample_idx, :]
                lane_ref_points = torch.cat([left_laneline, right_laneline], axis=-2).contiguous().detach()

                lane_ref_points[..., 0] = (lane_ref_points[..., 0] - self.pc_range[0]) / (self.pc_range[3] - self.pc_range[0])
                lane_ref_points[..., 1] = (lane_ref_points[..., 1] - self.pc_range[1]) / (self.pc_range[4] - self.pc_range[1])
                if self.pts_dim == 3:
                    lane_ref_points[..., 2] = (lane_ref_points[..., 2] - self.pc_range[2]) / (self.pc_range[5] - self.pc_range[2])

            output = output.permute(1, 0, 2)
            if self.return_intermediate:
                intermediate.append(output)
                intermediate_reference_points.append(reference_points)

        if self.return_intermediate:
            return torch.stack(intermediate), torch.stack(intermediate_reference_points)
        return output, reference_points


@TRANSFORMER_LAYER.register_module()
class CustomDetrTransformerDecoderLayer(BaseTransformerLayer):
    """Custom DETR transformer decoder layer for LaneSegNet."""

    def __init__(self, attn_cfgs, ffn_cfgs, operation_order=None, norm_cfg=dict(type='LN'), **kwargs):
        super(CustomDetrTransformerDecoderLayer, self).__init__(
            attn_cfgs=attn_cfgs, ffn_cfgs=ffn_cfgs, operation_order=operation_order,
            norm_cfg=norm_cfg, **kwargs)
        assert len(operation_order) == 6
        assert set(operation_order) == set(['self_attn', 'norm', 'cross_attn', 'ffn'])

print("Decoder components registered successfully!")

In [ ]:
# Dense Head - Lane Segmentation Head
from mmdet.models.dense_heads import AnchorFreeHead
from mmdet3d.core.bbox.coders import build_bbox_coder

@HEADS.register_module()
class LaneSegHead(AnchorFreeHead):
    """LaneSegNet Head for lane detection and segmentation."""

    def __init__(self,
                 num_classes,
                 in_channels,
                 num_query=200,
                 with_box_refine=False,
                 with_shared_param=None,
                 transformer=None,
                 bbox_coder=None,
                 num_reg_fcs=2,
                 code_weights=None,
                 bev_h=30,
                 bev_w=30,
                 pc_range=None,
                 pts_dim=3,
                 sync_cls_avg_factor=False,
                 num_lane_type_classes=3,
                 loss_cls=dict(type='CrossEntropyLoss', bg_cls_weight=0.1, use_sigmoid=False, loss_weight=1.0),
                 loss_bbox=dict(type='L1Loss', loss_weight=5.0),
                 loss_mask=dict(type='CrossEntropyLoss', loss_weight=3.0),
                 loss_dice=dict(type='DiceLoss', loss_weight=3.0),
                 loss_lane_type=dict(type='CrossEntropyLoss', use_sigmoid=True, loss_weight=1.0),
                 train_cfg=None,
                 test_cfg=None,
                 init_cfg=None,
                 pred_mask=False,
                 **kwargs):
        super(AnchorFreeHead, self).__init__(init_cfg)
        self.num_query = num_query
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.train_cfg = train_cfg or {}
        self.test_cfg = test_cfg or {}
        self.fp16_enabled = False
        self.loss_cls = build_loss(loss_cls)
        self.loss_bbox = build_loss(loss_bbox)
        self.loss_lane_type = build_loss(loss_lane_type)
        self.loss_mask = build_loss(loss_mask)
        self.loss_dice = build_loss(loss_dice)
        self.pred_mask = pred_mask

        if self.loss_cls.use_sigmoid:
            self.cls_out_channels = num_classes
        else:
            self.cls_out_channels = num_classes + 1
        
        self.transformer = build_transformer(transformer)
        self.embed_dims = self.transformer.embed_dims
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.pts_dim = pts_dim
        self.with_box_refine = with_box_refine
        self.with_shared_param = with_shared_param if with_shared_param is not None else not with_box_refine
        self.code_size = kwargs.get('code_size', pts_dim * 30)
        self.code_weights = nn.Parameter(torch.tensor(code_weights or [1.0] * self.code_size, requires_grad=False), requires_grad=False)
        self.bbox_coder = build_bbox_coder(bbox_coder) if bbox_coder else None
        self.pc_range = pc_range
        self.num_reg_fcs = num_reg_fcs
        self.num_lane_type_classes = num_lane_type_classes
        self._init_layers()

    def _init_layers(self):
        """Initialize prediction layers."""
        def _get_clones(module, N):
            return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])
        
        # Classification branch
        cls_branch = []
        for _ in range(self.num_reg_fcs):
            cls_branch.append(Linear(self.embed_dims, self.embed_dims))
            cls_branch.append(nn.LayerNorm(self.embed_dims))
            cls_branch.append(nn.ReLU(inplace=True))
        cls_branch.append(Linear(self.embed_dims, self.cls_out_channels))
        fc_cls = nn.Sequential(*cls_branch)

        # Regression branches
        reg_branch = []
        for _ in range(self.num_reg_fcs):
            reg_branch.append(Linear(self.embed_dims, self.embed_dims))
            reg_branch.append(nn.ReLU())
        reg_branch.append(Linear(self.embed_dims, self.code_size // 3))
        reg_branch = nn.Sequential(*reg_branch)

        reg_branch_offset = []
        for _ in range(self.num_reg_fcs):
            reg_branch_offset.append(Linear(self.embed_dims, self.embed_dims))
            reg_branch_offset.append(nn.ReLU())
        reg_branch_offset.append(Linear(self.embed_dims, self.code_size // 3))
        reg_branch_offset = nn.Sequential(*reg_branch_offset)

        # Query embedding
        self.query_embedding = nn.Embedding(self.num_query, self.embed_dims * 2)

        # Type branches
        cls_left_type_branch = []
        for _ in range(self.num_reg_fcs):
            cls_left_type_branch.append(Linear(self.embed_dims, self.embed_dims))
            cls_left_type_branch.append(nn.LayerNorm(self.embed_dims))
            cls_left_type_branch.append(nn.ReLU(inplace=True))
        cls_left_type_branch.append(Linear(self.embed_dims, self.num_lane_type_classes))
        fc_cls_left_type = nn.Sequential(*cls_left_type_branch)

        cls_right_type_branch = []
        for _ in range(self.num_reg_fcs):
            cls_right_type_branch.append(Linear(self.embed_dims, self.embed_dims))
            cls_right_type_branch.append(nn.LayerNorm(self.embed_dims))
            cls_right_type_branch.append(nn.ReLU(inplace=True))
        cls_right_type_branch.append(Linear(self.embed_dims, self.num_lane_type_classes))
        fc_cls_right_type = nn.Sequential(*cls_right_type_branch)

        # Mask branch
        mask_branch = nn.Sequential(
            nn.Linear(self.embed_dims, self.embed_dims),
            nn.ReLU(inplace=True),
            nn.Linear(self.embed_dims, self.embed_dims))

        num_pred = (self.transformer.decoder.num_layers + 1) if getattr(self, 'as_two_stage', False) else self.transformer.decoder.num_layers

        if not self.with_shared_param:
            self.cls_branches = _get_clones(fc_cls, num_pred)
            self.reg_branches = _get_clones(reg_branch, num_pred)
            self.reg_branches_offset = _get_clones(reg_branch_offset, num_pred)
            self.cls_left_type_branches = _get_clones(fc_cls_left_type, num_pred)
            self.cls_right_type_branches = _get_clones(fc_cls_right_type, num_pred)
            self.mask_embed = _get_clones(mask_branch, num_pred)
        else:
            self.cls_branches = nn.ModuleList([fc_cls for _ in range(num_pred)])
            self.reg_branches = nn.ModuleList([reg_branch for _ in range(num_pred)])
            self.reg_branches_offset = nn.ModuleList([reg_branch_offset for _ in range(num_pred)])
            self.cls_left_type_branches = nn.ModuleList([fc_cls_left_type for _ in range(num_pred)])
            self.cls_right_type_branches = nn.ModuleList([fc_cls_right_type for _ in range(num_pred)])
            self.mask_embed = nn.ModuleList([mask_branch for _ in range(num_pred)])

    def forward(self, mlvl_feats, bev_feats, img_metas):
        """Forward pass."""
        dtype = mlvl_feats[0].dtype
        object_query_embeds = self.query_embedding.weight.to(dtype)
        outputs = self.transformer(mlvl_feats, bev_feats, object_query_embeds, bev_h=self.bev_h, bev_w=self.bev_w,
                                   reg_branches=(self.reg_branches, self.reg_branches_offset) if self.with_box_refine else None,
                                   cls_branches=None, img_metas=img_metas)
        
        hs, init_reference, inter_references = outputs
        hs = hs.permute(0, 2, 1, 3)

        outputs_classes = []
        outputs_coord = []
        output_left_types = []
        output_right_types = []

        for lvl in range(hs.shape[0]):
            reference = init_reference if lvl == 0 else inter_references[lvl - 1]
            reference = inverse_sigmoid(reference)

            outputs_class = self.cls_branches[lvl](hs[lvl])
            output_left_type = self.cls_left_type_branches[lvl](hs[lvl])
            output_right_type = self.cls_right_type_branches[lvl](hs[lvl])

            tmp = self.reg_branches[lvl](hs[lvl])
            bs, num_query, _ = tmp.shape
            tmp = tmp.view(bs, num_query, -1, self.pts_dim)
            tmp = (tmp + reference).sigmoid()

            coord = tmp.clone()
            if self.pc_range is not None:
                coord[..., 0] = coord[..., 0] * (self.pc_range[3] - self.pc_range[0]) + self.pc_range[0]
                coord[..., 1] = coord[..., 1] * (self.pc_range[4] - self.pc_range[1]) + self.pc_range[1]
                if self.pts_dim == 3:
                    coord[..., 2] = coord[..., 2] * (self.pc_range[5] - self.pc_range[2]) + self.pc_range[2]
            centerline = coord.view(bs, num_query, -1).contiguous()

            offset = self.reg_branches_offset[lvl](hs[lvl])
            left_laneline = centerline + offset
            right_laneline = centerline - offset

            outputs_classes.append(outputs_class)
            outputs_coord.append(torch.cat([centerline, left_laneline, right_laneline], axis=-1))
            output_left_types.append(output_left_type)
            output_right_types.append(output_right_type)

        outs = {
            'all_cls_scores': torch.stack(outputs_classes),
            'all_lanes_preds': torch.stack(outputs_coord),
            'all_lanes_left_type': torch.stack(output_left_types),
            'all_lanes_right_type': torch.stack(output_right_types),
            'history_states': hs
        }
        return outs

print("LaneSegHead registered successfully!")

In [ ]:
# Main Model - LaneSegNet Detector

def build_bev_constructor(cfg):
    """Build BEV constructor from config."""
    if cfg is None:
        return None
    return build_backbone(cfg)

@DETECTORS.register_module()
class LaneSegNet(MVXTwoStageDetector):
    """LaneSegNet: Map Learning with Lane Segment Perception."""

    def __init__(self, bev_constructor=None, lane_head=None, lclc_head=None, 
                 bbox_head=None, lcte_head=None, video_test_mode=False, **kwargs):
        super(LaneSegNet, self).__init__(**kwargs)

        if bev_constructor is not None:
            self.bev_constructor = build_bev_constructor(bev_constructor)

        if lane_head is not None:
            train_cfg = self.train_cfg if self.train_cfg else {}
            lane_head.update(train_cfg=train_cfg.get('lane', {}))
            self.pts_bbox_head = build_head(lane_head)
        else:
            self.pts_bbox_head = None
        
        if lclc_head is not None:
            self.lclc_head = build_head(lclc_head)
        else:
            self.lclc_head = None

        if bbox_head is not None:
            train_cfg = self.train_cfg if self.train_cfg else {}
            bbox_head.update(train_cfg=train_cfg.get('bbox', {}))
            self.bbox_head = build_head(bbox_head)
        else:
            self.bbox_head = None

        if lcte_head is not None:
            self.lcte_head = build_head(lcte_head)
        else:
            self.lcte_head = None

        self.fp16_enabled = False
        self.video_test_mode = video_test_mode
        self.prev_frame_info = {'prev_bev': None, 'scene_token': None, 'prev_pos': 0, 'prev_angle': 0}

    def extract_img_feat(self, img, img_metas, len_queue=None):
        """Extract features of images."""
        B = img.size(0)
        if img is not None:
            if img.dim() == 5 and img.size(0) == 1:
                img.squeeze_()
            elif img.dim() == 5 and img.size(0) > 1:
                B, N, C, H, W = img.size()
                img = img.reshape(B * N, C, H, W)
            img_feats = self.img_backbone(img)

            if isinstance(img_feats, dict):
                img_feats = list(img_feats.values())
        else:
            return None
        
        if self.with_img_neck:
            img_feats = self.img_neck(img_feats)

        img_feats_reshaped = []
        for img_feat in img_feats:
            BN, C, H, W = img_feat.size()
            if len_queue is not None:
                img_feats_reshaped.append(img_feat.view(int(B/len_queue), len_queue, int(BN / B), C, H, W))
            else:
                img_feats_reshaped.append(img_feat.view(B, int(BN / B), C, H, W))
        return img_feats_reshaped

    @auto_fp16(apply_to=('img',))
    def extract_feat(self, img, img_metas=None, len_queue=None):
        """Extract features from images."""
        return self.extract_img_feat(img, img_metas, len_queue=len_queue)

    def forward(self, return_loss=True, **kwargs):
        """Forward method."""
        if return_loss:
            return self.forward_train(**kwargs)
        else:
            return self.forward_test(**kwargs)
    
    @auto_fp16(apply_to=('img',))
    def forward_train(self, img=None, img_metas=None, gt_lanes_3d=None, gt_lane_labels_3d=None,
                      gt_lane_adj=None, gt_lane_left_type=None, gt_lane_right_type=None,
                      gt_labels=None, gt_bboxes=None, gt_instance_masks=None, gt_bboxes_ignore=None):
        """Training forward pass."""
        len_queue = img.size(1)
        prev_img = img[:, :-1, ...]
        img = img[:, -1, ...]

        prev_bev = None
        img_metas = [each[len_queue-1] for each in img_metas]
        img_feats = self.extract_feat(img=img, img_metas=img_metas)
        bev_feats = self.bev_constructor(img_feats, img_metas, prev_bev)

        losses = dict()
        outs = self.pts_bbox_head(img_feats, bev_feats, img_metas)

        return losses

    def forward_test(self, img_metas, img=None, **kwargs):
        """Testing forward pass."""
        if not isinstance(img_metas, list):
            raise TypeError('img_metas must be a list')
        img = [img] if img is None else img
        return self.simple_test(img_metas, img, **kwargs)

    def simple_test(self, img_metas, img=None, rescale=False, **kwargs):
        """Simple test without augmentation."""
        img_feats = self.extract_feat(img=img, img_metas=img_metas)
        results_list = [dict() for i in range(len(img_metas))]
        return results_list

print("LaneSegNet main model registered successfully!")

# Training Configuration and Utilities

In [ ]:
# Training Utilities

def auto_scale_lr(cfg, distributed, logger):
    """Automatically scaling LR according to GPU number and sample per GPU."""
    if ('auto_scale_lr' not in cfg) or (not cfg.auto_scale_lr.get('enable', False)):
        if logger:
            logger.info('Automatic scaling of learning rate (LR) has been disabled.')
        return

    base_batch_size = cfg.auto_scale_lr.get('base_batch_size', None)
    if base_batch_size is None:
        return

    if distributed:
        _, world_size = get_dist_info()
        num_gpus = len(range(world_size))
    else:
        num_gpus = len(cfg.gpu_ids) if hasattr(cfg, 'gpu_ids') else 1

    samples_per_gpu = getattr(cfg.data.train, 'samples_per_gpu', 1)
    batch_size = num_gpus * samples_per_gpu
    if logger:
        logger.info(f'Training with {num_gpus} GPU(s) with {samples_per_gpu} samples per GPU. Total batch size is {batch_size}.')

    if batch_size != base_batch_size:
        scaled_lr = (batch_size / base_batch_size) * cfg.optimizer.lr
        if logger:
            logger.info(f'LR scaled from {cfg.optimizer.lr} to {scaled_lr}')
        cfg.optimizer.lr = scaled_lr


def create_work_dir(cfg):
    """Create work directory and save config."""
    mmcv.mkdir_or_exist(os.path.abspath(cfg.work_dir))
    cfg_filename = getattr(cfg, 'config_file', 'config.py')
    cfg.dump(os.path.join(cfg.work_dir, os.path.basename(cfg_filename)))


def get_logger(cfg):
    """Get root logger."""
    timestamp = time.strftime('%Y%m%d_%H%M%S', time.localtime())
    log_file = os.path.join(cfg.work_dir, f'{timestamp}.log')
    logger = get_root_logger(log_file=log_file, log_level=cfg.get('log_level', 20), name='lanesegnet')
    return logger

print("Training utilities defined successfully!")

In [ ]:
# Main Training Function

def train_lanesegnet(config_path, work_dir=None, resume_from=None, auto_resume=False, 
                     no_validate=False, gpu_id=0, seed=42, diff_seed=False, 
                     deterministic=False, autoscale_lr=False, cfg_options=None):
    """
    Train LaneSegNet model.
    
    Args:
        config_path: Path to config file
        work_dir: Working directory to save logs and models
        resume_from: Path to checkpoint to resume from
        auto_resume: Resume from latest checkpoint automatically
        no_validate: Whether not to validate during training
        gpu_id: GPU ID to use
        seed: Random seed
        diff_seed: Set different seed for different ranks
        deterministic: Set deterministic options for CUDNN
        autoscale_lr: Automatically scale lr
        cfg_options: Config options to override
    """
    # Load config
    cfg = Config.fromfile(config_path)
    if cfg_options is not None:
        cfg.merge_from_dict(cfg_options)
    
    # Setup multi-process
    setup_multi_processes(cfg)

    # Set cudnn_benchmark
    if cfg.get('cudnn_benchmark', False):
        torch.backends.cudnn.benchmark = True

    # Work directory setup
    if work_dir is not None:
        cfg.work_dir = work_dir
    elif cfg.get('work_dir', None) is None:
        cfg.work_dir = os.path.join('./work_dirs', os.path.splitext(os.path.basename(config_path))[0])
    
    if resume_from is not None:
        cfg.resume_from = resume_from
    if auto_resume:
        cfg.auto_resume = auto_resume

    cfg.gpu_ids = [gpu_id]

    if autoscale_lr:
        if 'auto_scale_lr' in cfg and 'base_batch_size' in cfg.auto_scale_lr:
            cfg.auto_scale_lr.enable = True

    # Initialize distributed environment
    distributed = False

    # Create work directory
    create_work_dir(cfg)
    
    # Initialize logger
    logger = get_logger(cfg)
    
    # Log environment info
    env_info_dict = collect_env()
    env_info = '\n'.join([(f'{k}: {v}') for k, v in env_info_dict.items()])
    logger.info('Environment info:\n' + '-'*60 + '\n' + env_info + '\n' + '-'*60)
    
    # Set random seeds
    seed_val = init_random_seed(seed)
    logger.info(f'Set random seed to {seed_val}, deterministic: {deterministic}')
    set_random_seed(seed_val, deterministic=deterministic)
    cfg.seed = seed_val
    
    # Build model and datasets
    logger.info('Building model...')
    model = build_model(cfg.model, train_cfg=cfg.get('train_cfg'), test_cfg=cfg.get('test_cfg'))
    model.init_weights()
    logger.info(f'Model:\n{model}')
    
    logger.info('Building datasets...')
    datasets = [build_dataset(cfg.data.train)]
    
    if len(cfg.get('workflow', [['train', 1]])) == 2:
        from copy import deepcopy
        val_dataset = deepcopy(cfg.data.val)
        if 'dataset' in cfg.data.train:
            val_dataset.pipeline = cfg.data.train.dataset.pipeline
        else:
            val_dataset.pipeline = cfg.data.train.pipeline
        val_dataset.test_mode = False
        datasets.append(build_dataset(val_dataset))
    
    # Setup checkpoint config metadata
    if cfg.checkpoint_config is not None:
        cfg.checkpoint_config.meta = dict(
            mmdet_version=mmdet_version,
            mmseg_version=mmseg_version,
            mmdet3d_version=mmdet3d_version,
            config=cfg.pretty_text,
            CLASSES=datasets[0].CLASSES,
            PALETTE=getattr(datasets[0], 'PALETTE', None)
        )
    
    model.CLASSES = datasets[0].CLASSES
    
    # Auto-scale lr
    auto_scale_lr(cfg, distributed=distributed, logger=logger)
    
    # Train the model
    logger.info('Starting training...')
    train_model(
        model,
        datasets,
        cfg,
        distributed=distributed,
        validate=(not no_validate),
        timestamp=time.strftime('%Y%m%d_%H%M%S', time.localtime()),
        meta=dict(env_info=env_info, seed=seed_val, exp_name=os.path.basename(config_path))
    )
    logger.info('Training finished!')

print("Main training function defined successfully!")

# Usage Instructions

To train the LaneSegNet model, use the `train_lanesegnet()` function with a config file path.

## Example Usage:

```python
# Train with default settings
train_lanesegnet(
    config_path='projects/configs/lanesegnet_r50_8x1_24e_olv2_subset_A.py',
    work_dir='./work_dirs/lanesegnet_r50',
    gpu_id=0,
    seed=42
)
```

## Parameters:
- `config_path`: Path to the config file
- `work_dir`: Directory to save logs and checkpoints
- `resume_from`: Resume training from a checkpoint
- `auto_resume`: Automatically resume from latest checkpoint
- `no_validate`: Skip validation during training
- `gpu_id`: GPU device ID to use
- `seed`: Random seed for reproducibility
- `diff_seed`: Use different seeds for different processes
- `deterministic`: Enable deterministic mode for CUDNN
- `autoscale_lr`: Automatically scale learning rate based on batch size
- `cfg_options`: Dict of config options to override

In [ ]:
# 🚀 Training Execution Example
print('='*80)
print('LANESEGNET TRAINING - READY TO RUN')
print('='*80)
print('\n✅ Environment setup complete')
print('✅ All models and utilities loaded')
print('\n📋 To start training, run:')
print()
train_lanesegnet(
    config_path='/content/LaneSegNet/projects/configs/lanesegnet_r50_8x1_24e_olv2_subset_A.py',
    work_dir='/content/work_dirs/lanesegnet_r50',
    gpu_id=0,
    seed=42,
    autoscale_lr=True
)
\n⚠️  Prerequisites:
1. Run cells 1.1-1.6 for environment setup
2. Run cell 2.1 to import all modules
3. Ensure dataset exists or is mounted from Google Drive


# 📊 Training Monitoring

## View GPU Status
```python
!nvidia-smi  # Check GPU usage
!nvidia-smi -l 1  # Continuous updates
```

## Check Training Logs
```bash
# View latest logs
!tail -50 /content/work_dirs/lanesegnet_r50/latest.log

# Watch logs in real-time
!tail -f /content/work_dirs/lanesegnet_r50/latest.log
```

## Troubleshooting

| Problem | Solution |
|---------|----------|
| Out of Memory | Reduce `samples_per_gpu=1` in config, use gradient checkpointing |
| Training too slow | Check GPU with `!nvidia-smi`, reduce `workers_per_gpu` |
| Data not found | Mount Google Drive or download dataset first |
| Module errors | Restart kernel and re-run env setup (cells 1.1-1.6) |

## Tips for Colab
- Models save to `/content/work_dirs/` - download before session ends
- Enable "Run all" and let training run overnight
- Monitor metrics with TensorBoard if integrated
- Use `resume_from` parameter to continue from checkpoints


In [ ]:
# Quick Commands for Monitoring During Training

import os

# GPU Status
print('GPU Status:')
!nvidia-smi --query-gpu=name,memory.total,memory.used,temperature.gpu --format=csv,noheader | head -1

# Training directory contents
work_dir = '/content/work_dirs/lanesegnet_r50'
if os.path.exists(work_dir):
    print(f'\nCheckpoints in {work_dir}:')
    files = sorted([f for f in os.listdir(work_dir) if f.endswith('.pth')])
    for f in files[-3:]:  # Show last 3 checkpoints
        size = os.path.getsize(os.path.join(work_dir, f)) / 1e9  # Size in GB
        print(f'  {f}: {size:.2f} GB')
else:
    print(f'\nTraining directory not yet created: {work_dir}')
    print('It will be created once training starts.')


In [ ]:
# 🚀 Training Execution Example
print('='*80)
print('LANESEGNET TRAINING - READY TO RUN')
print('='*80)
print('\n✅ Environment setup complete')
print('✅ All models and utilities loaded')
print('\n📋 To start training, run:')
print()
train_lanesegnet(
    config_path='/content/LaneSegNet/projects/configs/lanesegnet_r50_8x1_24e_olv2_subset_A.py',
    work_dir='/content/work_dirs/lanesegnet_r50',
    gpu_id=0,
    seed=42,
    autoscale_lr=True
)
# \n⚠️  Prerequisites:
# 1. Run cells 1.1-1.6 for environment setup
# 2. Run cell 2.1 to import all modules
# 3. Ensure dataset exists or is mounted from Google Drive


# 📊 Training Monitoring

## View GPU Status
```python
!nvidia-smi  # Check GPU usage
!nvidia-smi -l 1  # Continuous updates
```

## Check Training Logs
```bash
# View latest logs
!tail -50 /content/work_dirs/lanesegnet_r50/latest.log

# Watch logs in real-time
!tail -f /content/work_dirs/lanesegnet_r50/latest.log
```

## Troubleshooting

| Problem | Solution |
|---------|----------|
| Out of Memory | Reduce `samples_per_gpu=1` in config, use gradient checkpointing |
| Training too slow | Check GPU with `!nvidia-smi`, reduce `workers_per_gpu` |
| Data not found | Mount Google Drive or download dataset first |
| Module errors | Restart kernel and re-run env setup (cells 1.1-1.6) |

## Tips for Colab
- Models save to `/content/work_dirs/` - download before session ends
- Enable "Run all" and let training run overnight
- Monitor metrics with TensorBoard if integrated
- Use `resume_from` parameter to continue from checkpoints


In [ ]:
# Quick Commands for Monitoring During Training

import os

# GPU Status
print('GPU Status:')
!nvidia-smi --query-gpu=name,memory.total,memory.used,temperature.gpu --format=csv,noheader | head -1

# Training directory contents
work_dir = '/content/work_dirs/lanesegnet_r50'
if os.path.exists(work_dir):
    print(f'\nCheckpoints in {work_dir}:')
    files = sorted([f for f in os.listdir(work_dir) if f.endswith('.pth')])
    for f in files[-3:]:  # Show last 3 checkpoints
        size = os.path.getsize(os.path.join(work_dir, f)) / 1e9  # Size in GB
        print(f'  {f}: {size:.2f} GB')
else:
    print(f'\nTraining directory not yet created: {work_dir}')
    print('It will be created once training starts.')
